In [1]:
# Imports and globals setup
import argparse
import os, sys
import numpy as np
import pandas as pd
import time
import gc
import re
import random

import transformers
import torch

from transformers import LlamaForCausalLM, LlamaTokenizerFast

# 1) Get data from mmlu dev set
# 2) Collect baseline for 0 shot QA.
# 3) Collect info for "naive" paraphrase and "naive" question inoculation.

KEYPATH = ""
with open(KEYPATH, 'r') as f:
    API_KEY = f.readline()
    API_KEY = API_KEY.rstrip('\n')

print("Done")

2025-02-14 15:29:31.609867: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-14 15:29:31.624451: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-14 15:29:31.648003: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-14 15:29:31.648040: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-14 15:29:31.663500: I tensorflow/core/platform/cpu_feature_gua

Done


In [7]:
# Model selection

# TEST_MODEL = "Llama-3.2-3B"
# MODEL_REPO = "meta-llama/Llama-3.2-3B"
TEST_MODEL = "Llama-3.2-3B-Instruct"
MODEL_REPO = "meta-llama/Llama-3.2-3B-Instruct"  # Other model does not have a chat template; no chat support?

# JL - testing 1B for reference?
# TEST_MODEL = "Llama-3.2-1B-Instruct"
# MODEL_REPO = "meta-llama/Llama-3.2-1B-Instruct"  # Other model does not have a chat template; no chat support?

# larger Llama-3 model:
# TEST_MODEL = "Llama-3.1-8B-Instruct"
# MODEL_REPO = "meta-llama/Llama-3.1-8B-Instruct"  # Other model does not have a chat template; no chat support?

# Go bigger?


DEVICE_STR = 'cuda'
# DEVICE_STR = 'cpu'
# DEVICE_STR = 'auto'

# For local code testing.
# TEST_MODEL = "TinyLlama_v1.1"
# MODEL_REPO = "TinyLlama/TinyLlama_v1.1"

# Local testing.
# TEST_MODEL = "TinyLlama-1.1B-Chat-v1.0"
# MODEL_REPO = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Model card: https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0
# It was tuned on this: https://huggingface.co/datasets/stingning/ultrachat
# Further tuned for preference on this: https://huggingface.co/datasets/openbmb/UltraFeedback

GEN_TEMP = 1.0  # used for all generations for now.

# For short answers
MAX_NEW_TOKENS = 10

# For paraphrasing.
MAX_PARAPHRASE_TOKENS = 1000  # Roughly 2x the number of tokens for the longest question.

CHOICES = ["A", "B", "C", "D"]

BATCH_SIZE = 1  # one at a time for now. Avoids padding issues.

# Sometimes the model doesn't listen, and outputs stuff that doesn't follow the specified format
# This dictates how patient we'll be!
PARSE_RETRIES = 5

DATA_DIR = "/home/jlim/MS_Project/MMLU_Baseline/data"

ANSWER_STRING_MAP = {'0': "A", '1': "B", '2': "C", '3': "D"}

print("Data directory: " + str(DATA_DIR))

Data directory: /home/jlim/MS_Project/MMLU_Baseline/data


In [3]:
# Prepare dev data, supporting functions.


# read all questions
subjects = sorted(
    [f.split("_test.csv")[0] for f in os.listdir(os.path.join(DATA_DIR, "test")) if "_test.csv" in f])

dev_df = None
for subject in subjects:
    # TODO: Separate by subject?
    if dev_df is None:
        dev_df = pd.read_csv(os.path.join(DATA_DIR, "dev", subject + "_dev.csv"), header=None)
    else:
        dev_df = pd.concat([dev_df, pd.read_csv(os.path.join(DATA_DIR, "dev", subject + "_dev.csv"), header=None)])
        
num_qs = dev_df.shape[0]
print("Dev dataset size: " + str(num_qs))

# Define support functions.
def ask_q(pipeline, question_list):
    # Ask the Q, get the response.
    # Use llama prompt format, though with no examples.
    prompt_turn_string = ("Given the following question and four candidate answers (A, B, C and D), choose the best answer.\nQuestion: {}\n"
                          "Your response should end with \"The best answer is [the_answer_letter]\" where the [the_answer_letter] is one of A, B, C or D.")

    # Only used for prompting with examples.
    # response_template = "\n\nThe best answer is {}."

    promptlist = []
    for q in question_list:

        # return the choice string.
        msgs = [{"role": "user", "content": prompt_turn_string.format(q)}]

        # Return the completed template.
        prompt = pipeline.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        promptlist.append(prompt)
    # print("Prompt:")
    # print(prompt)

    # Encode a batch to be processed at once.
    # Dealing with padding: https://huggingface.co/docs/transformers/en/pad_truncation
    encoded = pipeline.tokenizer(promptlist, return_tensors="pt", padding=False)  # padding=True

    # TEST: can you batch?
    # encoded = pipeline.tokenizer([prompt, prompt], return_tensors="pt")

    # Put on proper device?
    # Inspiration: https://discuss.huggingface.co/t/device-map-auto-with-error-expected-all-tensors-to-be-on-the-same-device/31938/6
    encoded.to(DEVICE_STR)


    generated_output = pipeline.model.generate(**encoded, do_sample=True, temperature=GEN_TEMP, max_new_tokens=MAX_NEW_TOKENS,
                                      return_dict_in_generate=True, output_scores=True, output_logits=True)

    # Free memory
    del encoded

    respset = []
    for i in range(len(question_list)):
        # Subject to MAX_NEW_TOKENS limit
        respset.append(pipeline.tokenizer.decode(generated_output.sequences[i][-MAX_NEW_TOKENS:]))

    return respset


# Modified MMLU format function.
def format_question_single(df, idx, include_answer=True):
    prompt = df.iloc[idx, 0]
    k = df.shape[1] - 2  # answer index.
    for j in range(k):
        prompt += "\n{}. {}".format(CHOICES[j], df.iloc[idx, j+1])

    if include_answer:
        prompt += "\nAnswer:"
        prompt += " {}\n\n".format(df.iloc[idx, k + 1])
    return prompt


Dev dataset size: 285


In [8]:
# Model, tokenizer, pipeline init
device_map_setting = DEVICE_STR # 'auto'  # 'cuda'

model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_REPO,
    token=API_KEY,
    device_map=device_map_setting
)

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_REPO, token=API_KEY)

if torch.cuda.is_available():
    print("GPU available...")
else:
    print("No GPU available...")

pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    torch_dtype=torch.float16,  # Probably won't work on cpu...
    tokenizer=tokenizer,
    device_map=device_map_setting,
)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda


GPU available...


In [ ]:

# Prompt gen test
# Basically, how well the model deals with different prompts.
rand_idx = random.choice(range(dev_df.shape[0]))

question_no_choices = dev_df.iloc[rand_idx, 0]

question_str = format_question_single(dev_df, rand_idx, include_answer=False)

print("Test question: " + str(question_str))


# Test paraphrase
# Not using the word "paraphrase" here.
# 1
paraphrase_prompt = ("Please rewrite the following question using different words. "
                     "The rewritten question should have the same meaning as the original. "
                     "Your response should be enclosed in double quotes like the following format: \"\"<new question>\"\".\nOriginal Question: {}\n")

# Order change
# 2
# paraphrase_prompt = ("Your response should be enclosed in double quotes like the following format: \"\"<new question>\"\"."
#                      "Given this, please rewrite the following question using different words. "
#                      "The rewritten question should have the same meaning as the original. "
#                      "\nOriginal Question: {}\n")

# 3
# Guessing at a task used in this dataset; it's mentioned on the dataset card:
# https://huggingface.co/datasets/stingning/ultrachat
# paraphrase_prompt = ("Can you help me reword this exam question, so that my students can understand it better? "
#                         "Please put the reworded question in quotes so that I can use it. "
#                         "\nHere is the question to reword: \"{}\"\n")
# llama3 is much better... no comparison to tinyllama!
# Prompt 1 and 3 work fine for the most part.

# New prompt, related to prompt 1 but improving formatting because the double double quoting seems unusual.
paraphrase_prompt = ("Please rewrite the following question using different words. "
                     "The rewritten question should have the same meaning as the original. "
                     "Your response should be enclosed in double quotes like the following format: \"<new question>\".\nOriginal Question: \'{}\'\n")


# Debugging, dummy prompt:
# paraphrase_prompt = ("Please output the following string: \"\"hello world.\"\""

# Sometimes, you cannot separate the question and the answers!
paraphrase_msgs = [{"role": "user", "content": paraphrase_prompt.format(question_str)}]  # question_no_choices

successful_parse = False
fail_count = 0
para_q = None
while not successful_parse:

    prompt = pipeline.tokenizer.apply_chat_template(paraphrase_msgs, tokenize=False, add_generation_prompt=True)
    encoded_paraphrase_request = pipeline.tokenizer(prompt, return_tensors="pt", padding=False) # Turning padding off for now

    # Put on proper device?
    # Inspiration: https://discuss.huggingface.co/t/device-map-auto-with-error-expected-all-tensors-to-be-on-the-same-device/31938/6
    encoded_paraphrase_request.to(DEVICE_STR)


    generated_output = pipeline.model.generate(**encoded_paraphrase_request, do_sample=True, temperature=GEN_TEMP, max_new_tokens=MAX_PARAPHRASE_TOKENS,
                                        return_dict_in_generate=True, output_scores=True, output_logits=True)

    generated_output = generated_output.sequences[0][-MAX_PARAPHRASE_TOKENS:]

    # Detokenize
    generated_output =pipeline.tokenizer.decode(generated_output)

    # Keep tensor memory clear!
    del encoded_paraphrase_request

    # result = re.search("\"\".*\"\"", generated_output)

    # The prompt has a string that matches this. Ignore this by getting only the last match!
    # all_matches = re.findall("\"\".*\"\"", generated_output)
    all_matches = re.findall("\".*\"", generated_output)  # Switching to single double quotes
    all_matches = all_matches[1:]  # Exclude first match

    # Just look at the last match.
    if len(all_matches) == 0:
        result = None
    else:
        result = all_matches[-1]

    if result is not None:
        successful_parse = True
        # para_q = generated_output[result.start():result.end()]
        para_q = result
        para_q = para_q.strip("\"")

        print("Paraphrase response:")
        print(generated_output)
        print("~~~~~~~~~~~~~~~~~~~~~")

        print("Paraphrased question:")
        print(para_q)
    else:
        fail_count += 1
        print("Failing Paraphrase response:")
        print(generated_output)
        if fail_count > PARSE_RETRIES:
            # Fallback to not paraphrasing?
            print("Could not generate a valid paraphrase response!")
            break

# Okay, remember this example. It's MC but it's a completion task!

# The pleura
# A. have no sensory innervation.
# B. are separated by a 2 mm space.
# C. extend into the neck.
# D. are composed of respiratory epithelium.

# Note 2: Sometimes it ignores the answers, sometimes it doesn't


NameError: name 'random' is not defined

: 

In [26]:
# Defining paraphrasing strategies to test.
def ask_q(pipeline, question_list):
    # Ask the Q, get the response.
    # Use llama prompt format, though with no examples.
    prompt_turn_string = ("Given the following question and four candidate answers (A, B, C and D), choose the best answer.\nQuestion: {}\n"
                          "Your response should end with \"The best answer is [the_answer_letter]\" where the [the_answer_letter] is one of A, B, C or D.")

    # Only used for prompting with examples.
    # response_template = "\n\nThe best answer is {}."

    promptlist = []
    for q in question_list:

        # return the choice string.
        msgs = [{"role": "user", "content": prompt_turn_string.format(q)}]

        # Return the completed template.
        prompt = pipeline.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        promptlist.append(prompt)
    # print("Prompt:")
    # print(prompt)

    # Encode a batch to be processed at once.
    # Dealing with padding: https://huggingface.co/docs/transformers/en/pad_truncation
    encoded = pipeline.tokenizer(promptlist, return_tensors="pt", padding=False) # 

    # TEST: can you batch?
    # encoded = pipeline.tokenizer([prompt, prompt], return_tensors="pt")

    # Put on proper device?
    # Inspiration: https://discuss.huggingface.co/t/device-map-auto-with-error-expected-all-tensors-to-be-on-the-same-device/31938/6
    encoded.to(DEVICE_STR)

    # Reducing output spam: https://stackoverflow.com/questions/69609401/suppress-huggingface-logging-warning-setting-pad-token-id-to-eos-token-id
    generated_output = pipeline.model.generate(**encoded, do_sample=True, temperature=GEN_TEMP, max_new_tokens=MAX_NEW_TOKENS,
                                      return_dict_in_generate=True, output_scores=True, output_logits=True, pad_token_id=tokenizer.eos_token_id)

    # Free memory
    del encoded

    respset = []
    for i in range(len(question_list)):
        # Subject to MAX_NEW_TOKENS limit
        respset.append(pipeline.tokenizer.decode(generated_output.sequences[i][-MAX_NEW_TOKENS:]))

    return respset

def no_paraphrase_strategy(pipeline, question, dev_df, idx):
    return question # Surprising!


# Basic, zero shot paraphrases. Expecting poor results.
def naive_paraphrase_strategy(pipeline, question, dev_df, idx):

    orig_q_str = dev_df.iloc[idx, 0]

    # Not using the word "paraphrase" here.
    # 1
    # paraphrase_prompt = ("Please rewrite the following question using different words. "
    #                      "The rewritten question should have the same meaning as the original. "
    #                      "Your response should be enclosed in double quotes like the following format: \"\"<new question>\"\".\nOriginal Question: {}\n")

    # Order change
    # 2
    # paraphrase_prompt = ("Your response should be enclosed in double quotes like the following format: \"\"<new question>\"\"."
    #                      "Given this, please rewrite the following question using different words. "
    #                      "The rewritten question should have the same meaning as the original. "
    #                      "\nOriginal Question: {}\n")

    # 3
    # Guessing at a task used in this dataset; it's mentioned on the dataset card:
    # https://huggingface.co/datasets/stingning/ultrachat
    paraphrase_prompt = ("Can you help me reword this exam question, so that my students can understand it better? "
                         "Please put the reworded question in quotes at the beginning of your response so that I can use it. "
                         "\nHere is the question to reword: \'{}\'\n")
    # Still not obeying quotes rule.

    # Debugging, dummy prompt:
    # paraphrase_prompt = ("Please output the following string: \"\"hello world.\"\""

    # Need to put the full question!
    paraphrase_msgs = [{"role": "user", "content": paraphrase_prompt.format(orig_q_str)}] # orig_q_str # question
    # Implicit completion questions?


    print("Original Question~~~~:")
    print(question)

    MAX_VERIFY_RETRIES = 5
    verify_retries = 0

    while True:

        successful_parse = False
        fail_count = 0
        para_q = None
        while not successful_parse:

            prompt = pipeline.tokenizer.apply_chat_template(paraphrase_msgs, tokenize=False, add_generation_prompt=True)
            encoded_paraphrase_request = pipeline.tokenizer(prompt, return_tensors="pt", padding=False)

            # Put on proper device?
            # Inspiration: https://discuss.huggingface.co/t/device-map-auto-with-error-expected-all-tensors-to-be-on-the-same-device/31938/6
            encoded_paraphrase_request.to(DEVICE_STR)


            generated_output = pipeline.model.generate(**encoded_paraphrase_request, do_sample=True, temperature=GEN_TEMP, max_new_tokens=MAX_PARAPHRASE_TOKENS,
                                            return_dict_in_generate=True, output_scores=True, output_logits=True, pad_token_id=tokenizer.eos_token_id)


            # Update to exclude prompt length?
            # https://discuss.huggingface.co/t/generate-returns-full-prompt-plus-answer/70453

            generated_output = generated_output.sequences[0][encoded_paraphrase_request['input_ids'].shape[1]:]
            # generated_output = generated_output.sequences[0][-MAX_PARAPHRASE_TOKENS:]

            # Detokenize
            generated_output = pipeline.tokenizer.decode(generated_output)

            # Keep tensor memory clear!
            del encoded_paraphrase_request

            # result = re.search("\"\".*\"\"", generated_output)

            # The prompt has a string that matches this. Ignore this by getting only the last match!
            # all_matches = re.findall("\"\".*\"\"", generated_output)
            all_matches = re.findall("\".*\"", generated_output)  # Switching to single quotes

            # all_matches = all_matches[1:]  # Exclude first match

            # # Just look at the last match.
            # if len(all_matches) == 0:
            #     result = None
            # else:
            #     result = all_matches[-1]

            # Just look at the first match.
            if len(all_matches) == 0:
                result = None
            else:
                result = all_matches[0]

            if result is not None:
                successful_parse = True
                # para_q = generated_output[result.start():result.end()]
                para_q = result
                para_q = para_q.strip("\"")

                # print("Paraphrase response:")
                # print(generated_output)
                # print("~~~~~~~~~~~~~~~~~~~~~")



                print("Paraphrased question~~~~:")
                print(para_q)
            else:
                fail_count += 1
                # print("Failing Paraphrase response:")
                # print(generated_output)
                if fail_count > PARSE_RETRIES:
                    # Fallback to not paraphrasing?
                    print("Could not generate a valid paraphrase response!")
                    return None


        # TODO: Swap ordering, get rid of positional bias.
        verify_prompt = ("Do the following two questions have the same meaning? "
                        "You must answer must start with either \"Yes\" or \"No\"."
                        "\nQuestion 1: '{q1}'"
                        "\nQuestion 2: '{q2}'\n")

        # Alternate... harder or easier?
        # verify_prompt = ("Do the following two questions have the same answer? "
        #                  "Your answer must begin with either \"Yes\" or \"No\"."
        #                  "\nQuestion 1: {q1}"
        #                  "\nQuestion 2: {q1}\n")

        verify_msgs = [{"role": "user", "content": verify_prompt.format(q1=orig_q_str, q2=para_q)}]

        successful_parse = False
        fail_count = 0
        while not successful_parse:

            prompt = pipeline.tokenizer.apply_chat_template(verify_msgs, tokenize=False, add_generation_prompt=True)
            encoded_verify_request = pipeline.tokenizer(prompt, return_tensors="pt", padding=False) # Keeping padding off for now.

            # Put on proper device?
            # Inspiration: https://discuss.huggingface.co/t/device-map-auto-with-error-expected-all-tensors-to-be-on-the-same-device/31938/6
            encoded_verify_request.to(DEVICE_STR)


            generated_output = pipeline.model.generate(**encoded_verify_request, do_sample=True, temperature=GEN_TEMP, max_new_tokens=MAX_PARAPHRASE_TOKENS,
                                            return_dict_in_generate=True, output_scores=True, output_logits=True, pad_token_id=tokenizer.eos_token_id)

            # generated_output = generated_output.sequences[0][-MAX_NEW_TOKENS:]
            generated_output = generated_output.sequences[0][encoded_verify_request['input_ids'].shape[1]:]

            # Detokenize, conver to lowercase.
            generated_output = pipeline.tokenizer.decode(generated_output).lower()

            # Keep tensor memory clear!
            del encoded_verify_request

            if re.search("^yes", generated_output):
                # yes
                # "Trust" the llm
                # print the response
                print("Validation response~~~~:")
                print(generated_output)
                return para_q
            elif re.search("^no", generated_output):
                # no
                print("Validation response~~~~:")
                print(generated_output)

                verify_retries+= 1
                # Will try the whole process again!
                break
                if verify_retries > MAX_VERIFY_RETRIES:
                    print("Could not verify a similar response!")
                    return None

            else:
                fail_count += 1
                print("Failing Validation response:")
                print(generated_output)
                if fail_count > PARSE_RETRIES:
                    print("Could not generate a valid response!")
                    return None

    return None

# Choose our strategy

# QUESTION_STRATEGY = no_paraphrase_strategy
QUESTION_STRATEGY = naive_paraphrase_strategy

print("Current question strategy: " + str(QUESTION_STRATEGY.__name__))


Current question strategy: naive_paraphrase_strategy


In [ ]:
# MMLU dev set. Collect dev set data
avg_runs = 1

avg_accs = []
avg_props_parsed = []
avg_parsed_accs = []

# Change stdout so as not to spam jupyter console... seems to be causing issues
# https://stackoverflow.com/questions/4675728/redirect-stdout-to-a-file-in-python
print("Starting data collection...")

old_stdout = sys.stdout
# with open("/home/jlim/MS_Project/paraphrase_validate/run_out.txt", 'w') as sys.stdout:



for a in range(avg_runs):

    num_answered = 0
    num_correct_and_answered = 0

    num_correct = 0
    num_qs = dev_df.shape[0]
    num_parse_fail = 0  # record how many times the model did not generate a parseable answer.

    q_index_ordering = list(range(num_qs))
    random.shuffle(q_index_ordering)

    # ???
    # answers = CHOICES[:dev_df.shape[1]-2]
    print("Dev dataset size: " + str(num_qs))

    # Batch question processing.
    num_batches = int(num_qs / BATCH_SIZE)
    leftover = num_qs % BATCH_SIZE

    iter_batches = num_batches
    if leftover != 0:
        iter_batches += 1 # Should work...

    for i in range(iter_batches):
        # print(str(i*BATCH_SIZE) + "/" + str(num_qs))

        q_list = []
        answer_list = []

        # Manage leftover bit.
        upper_end = (i+1)*BATCH_SIZE
        if upper_end > num_qs:
            upper_end = num_qs

        for j in range(i*BATCH_SIZE, upper_end):
            question = format_question_single(dev_df, j, include_answer=False)
            answer = dev_df.iloc[j, dev_df.shape[1] - 1] # Dunno why the other code had -2? Dev perhaps is different?

            # Switch to whatever question strategy I want...
            question = QUESTION_STRATEGY(pipeline, question, dev_df, i)
            q_list.append(question)
            answer_list.append(answer)

            # print("The function name: " + QUESTION_STRATEGY.__name__)

            # print("Question:")
            # print(question)

            # print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
            # print("Response:")
            # print(resp)
            # print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

        resps = ask_q(pipeline, q_list)

        for idx, resp in enumerate(resps):
            # parse for answer.
            # Logic for checking answer.
            pred = CHOICES[0]  # Default to "A"
            updated = False
            for option in CHOICES:
                # match_str = "The best answer is {}.".format(option)
                if option in resp:
                    pred = option
                    updated = True

            if not updated:
                num_parse_fail += 1
                # print("Warning: no answer found. Artificially selecting \"{}\".".format(pred))
                # print("Question:")
                # print(question)       

                # print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
                # print("Response:")
                # print(resp)
                # print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
            else:
                num_answered += 1

            if pred == answer_list[idx]:
                num_correct += 1
                if updated:
                    num_correct_and_answered += 1

    print("Run # " + str(a + 1) + "; Stats:")
    total_accuracy = num_correct / num_qs
    print("Overall accuracy: " + str(total_accuracy))
    total_parse_fails = num_parse_fail / num_qs
    print("Overall proportion of nonanswer/parse fails: " + str(total_parse_fails))
    print("Transpose: proportion parsed correctly: " + str(1.0 - total_parse_fails))
    total_answered_accuracy = num_correct_and_answered / num_answered
    print("Overall proportion of correct answers that parsed correctly: " + str(total_answered_accuracy))

    avg_accs.append(total_accuracy)
    avg_props_parsed.append(1.0 - total_parse_fails)
    avg_parsed_accs.append(total_answered_accuracy)
    # sys.stdout.flush()

print("Average accuracy of runs: " + str(np.mean(avg_accs)))
print("Average parse proportion of runs: " + str(np.mean(avg_props_parsed)))
print("Average accuracy for properly parsed answers for runs: " + str(np.mean(avg_parsed_accs)))

# Additional stats
print("Average accuracy of runs, std: " + str(np.std(avg_accs)))
print("Average parse proportion of runs, std: " + str(np.std(avg_props_parsed)))
print("Average accuracy for properly parsed answers for runs, std: " + str(np.std(avg_parsed_accs)))
print()
print("Average accuracy of runs, max-min: " + str(np.max(avg_accs)) + "-" + str(np.min(avg_accs)))
print("Average parse proportion of runs, max-min: " + str(np.max(avg_props_parsed)) + "-" + str(np.min(avg_props_parsed)))
print("Average accuracy for properly parsed answers for runs, max-min: " + str(np.max(avg_parsed_accs)) + "-" + str(np.min(avg_parsed_accs)))

sys.stdout = old_stdout
print("Done.")


Starting data collection...
Dev dataset size: 285
Original Question~~~~:
Find all c in Z_3 such that Z_3[x]/(x^2 + c) is a field.
A. 0
B. 1
C. 2
D. 3
Paraphrased question~~~~:
Describe the values of c in Z_3 such that the polynomial quotient Z_3[x] / (x^2 + c) results in a field, where Z_3 is the set of integers modulo 3.
Validation response~~~~:
no<|eot_id|>
Paraphrased question~~~~:
'Find all possible values of c in the set Z_3 (which means c can be 0, 1, or 2) such that when we divide the polynomial x^2 + c by x^2, the resulting quotient is a field (i.e., it has no zero divisors and has an additive and multiplicative identity)'.
Validation response~~~~:
no<|eot_id|>
Paraphrased question~~~~:
Identify the elements c in Z_3 (mod 3) such that when the polynomial x^2 + c is divided into the polynomial x in Z_3, the resulting quotient is a polynomial with no linear remainder.
Validation response~~~~:
no.<|eot_id|>
Paraphrased question~~~~:
Find all values of c in Z_3 such that when we di

KeyboardInterrupt: 